In [1]:
import os
import sys
import tkinter as tk
from tkinter import filedialog
import cv2
import numpy as np
import psycopg2
from pgvector.psycopg2 import register_vector
from deepface import DeepFace

DB_CONFIG = {
    "dbname": "FACEPROJECT",
    "user": "postgres",
    "password": os.environ.get("DB_PASSWORD", ""),  
    "host": "localhost",
    "port": 5432
}

MODEL_NAME = "ArcFace"            
DETECTOR_BACKEND = "retinaface"   
COSINE_THRESHOLD = 0.65           
FRAME_SKIP = 3                    

def get_file_via_dialog(title, file_types, multiple=False):
    """Fallback file picker dialog."""
    try:
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        if multiple:
            selected = filedialog.askopenfilenames(title=title, filetypes=file_types)
        else:
            selected = filedialog.askopenfilename(title=title, filetypes=file_types)
        root.destroy()
        return selected
    except Exception:
        return ""

def get_input_path(prompt_text, file_title, file_types, multiple=False):
    """
    Lets you either type/paste the file path directly in the terminal,
    or press ENTER to open the file chooser window.
    """
    print(f"\n{prompt_text}")
    print("  -> Type/Paste the file path directly, OR")
    print("  -> Press [ENTER] to open a file chooser window:")
    user_val = input("Path: ").strip().strip('"').strip("'")

    if user_val:
        if multiple:
            # Handles comma-separated paths if user pasted multiple
            return [p.strip() for p in user_val.split(",") if os.path.exists(p.strip())]
        return user_val if os.path.exists(user_val) else None

    # Fallback to GUI picker if user pressed Enter
    print("[*] Opening file dialog...")
    return get_file_via_dialog(file_title, file_types, multiple)


def query_postgres(cur, embedding):
    """Executes HNSW-indexed cosine distance search (<=>) in FACEPROJECT."""
    query = """
        SELECT id, name, age, gender, clothes_description, last_seen_location,
               face_embedding <=> %s AS distance
        FROM missing_persons
        WHERE face_embedding IS NOT NULL
        ORDER BY distance ASC
        LIMIT 1;
    """
    cur.execute(query, (embedding,))
    row = cur.fetchone()

    if row:
        dist = float(row[6])
        if dist <= COSINE_THRESHOLD:
            return {
                "id": row[0],
                "name": row[1],
                "age": row[2],
                "gender": row[3],
                "clothes": row[4],
                "last_seen": row[5],
                "distance": dist
            }, dist
        return None, dist

    return None, 1.0


def scan_frame(frame, cur):
    """Extracts all faces in frame and verifies each against FACEPROJECT."""
    results = []
    try:
        faces = DeepFace.extract_faces(
            img_path=frame,
            detector_backend=DETECTOR_BACKEND,
            enforce_detection=False,
            align=True
        )

        for face_obj in faces:
            area = face_obj["facial_area"]
            x, y, w, h = area["x"], area["y"], area["w"], area["h"]

            if w < 28 or h < 28:
                continue

            crop = frame[max(0, y):y + h, max(0, x):x + w]
            if crop.size == 0:
                continue

            reps = DeepFace.represent(
                img_path=crop,
                model_name=MODEL_NAME,
                detector_backend="skip",
                enforce_detection=False
            )
            raw_emb = reps[0]["embedding"]
            emb = np.array(raw_emb, dtype=np.float32)
            norm = np.linalg.norm(emb)
            if norm > 0:
                emb = emb / norm

            matched_person, dist = query_postgres(cur, emb)
            results.append((x, y, w, h, matched_person, dist))

    except Exception:
        pass

    return results


def annotate_frame(frame, detections):
    for x, y, w, h, person, dist in detections:
        if person:
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 0, 255), 3)
            header = f"ALERT: {person['name']} ({dist:.2f})"
            cv2.putText(frame, header, (x, max(22, y - 24)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 0, 255), 2)
            info_str = f"Age:{person['age']} | Clothes:{person['clothes']}"
            cv2.putText(frame, info_str, (x, max(40, y - 6)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 255), 1)
        else:
            cv2.rectangle(frame, (x, y), (x + w, y + h), (60, 60, 60), 1)
    return frame


def process_images(cur, image_paths):
    if isinstance(image_paths, str):
        image_paths = [image_paths]

    for path in image_paths:
        frame = cv2.imread(path)
        if frame is None:
            print(f"[!] Unable to read image: {path}")
            continue

        print(f"[*] Scanning image: {os.path.basename(path)}...")
        detections = scan_frame(frame, cur)
        frame = annotate_frame(frame, detections)

        base, fname = os.path.split(path)
        out_path = os.path.join(base, f"matched_{fname}")
        cv2.imwrite(out_path, frame)
        print(f"  [+] Match search finished. Output saved to:\n      {out_path}")


def process_video(cur, video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"[!] Could not open video file: {video_path}")
        return

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0

    base, fname = os.path.split(video_path)
    out_path = os.path.join(base, f"matched_{fname}")
    out = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    frame_count = 0
    cached_detections = []

    print(f"[*] Processing video: {fname} (Press 'q' in the window to finish early)")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1
        if frame_count % FRAME_SKIP == 0:
            cached_detections = scan_frame(frame, cur)

        frame = annotate_frame(frame, cached_detections)
        out.write(frame)
        cv2.imshow("Crowd Search Preview", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    out.release()
    cv2.destroyAllWindows()
    print(f"[+] Annotated video saved to:\n    {out_path}")


def main():
    try:
        conn = psycopg2.connect(**DB_CONFIG)
        register_vector(conn)
        cur = conn.cursor()
        print("[+] Connected to PostgreSQL (FACEPROJECT) successfully.")
    except Exception as e:
        print(f"[!] Database connection error: {e}")
        return

    # Clear terminal menu prompt
    print("\n" + "=" * 45)
    print("      FACE RECOGNITION SEARCH MENU")
    print("=" * 45)
    print("  [1] Scan a Crowd PHOTO (or photos)")
    print("  [2] Scan a Crowd VIDEO")
    print("  [3] Exit")
    print("=" * 45)

    choice = input("Enter your choice (1, 2, or 3): ").strip()

    if choice == "1":
        photos = get_input_path(
            "Select Crowd Photo(s):",
            "Choose Crowd Photo(s)",
            [("Image Files", "*.jpg *.jpeg *.png *.webp *.bmp"), ("All Files", "*.*")],
            multiple=True
        )
        if photos:
            process_images(cur, photos)
        else:
            print("[!] No photos selected.")

    elif choice == "2":
        video = get_input_path(
            "Select Crowd Video:",
            "Choose Crowd Video",
            [("Video Files", "*.mp4 *.avi *.mkv *.mov"), ("All Files", "*.*")],
            multiple=False
        )
        if video:
            process_video(cur, video)
        else:
            print("[!] No video selected.")

    elif choice == "3":
        print("[*] Exiting.")
    else:
        print("[!] Invalid selection. Please enter 1, 2, or 3.")

    cur.close()
    conn.close()


if __name__ == "__main__":
    main()


[+] Connected to PostgreSQL (FACEPROJECT) successfully.

      FACE RECOGNITION SEARCH MENU
  [1] Scan a Crowd PHOTO (or photos)
  [2] Scan a Crowd VIDEO
  [3] Exit


Enter your choice (1, 2, or 3):  1



Select Crowd Photo(s):
  -> Type/Paste the file path directly, OR
  -> Press [ENTER] to open a file chooser window:


Path:  


[*] Opening file dialog...
[*] Scanning image: WhatsApp Image 2026-09-25 at 9.00.02 AM.jpeg...
  [+] Match search finished. Output saved to:
      C:/Users/Syam Sundar/Downloads\matched_WhatsApp Image 2026-09-25 at 9.00.02 AM.jpeg
